# GNN Fraud Detection — Transaction Subgraph Visualization

This notebook demonstrates how the GNN-based fraud detection model processes incoming transactions.

The key insight: unlike a traditional ML model that looks at each transaction in isolation,
the GNN extracts a **2-hop subgraph** around each new transaction and uses the entire neighborhood
as context. This lets it detect **fraud rings** — groups of users and merchants behaving suspiciously
together, even if each individual transaction looks normal.

**Graph structure:**
- **Users** (card holders) — left side of bipartite graph
- **Merchants** — right side of bipartite graph
- **Transactions** — edges between users and merchants

**2-hop subgraph extraction for a new transaction `user → merchant`:**
1. **Anchor edge**: the new transaction
2. **1-hop merchants**: other merchants the same user has visited (transaction history)
3. **2-hop users**: other users who share those same merchants (potential fraud ring members)

**Sections:**
1. Graph Overview
2. Transaction Stream & Subgraph Extraction
3. Endpoint Inference
4. Results Visualization
5. Shapley Explanations

In [14]:
import pandas as pd
import numpy as np
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from collections import defaultdict
import boto3
import json
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

Imports OK


In [15]:
import os

# --- Paths ---
PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
DATA_DIR    = os.path.join(PROJECT_DIR, 'data', 'TabFormer', 'gnn')
TEST_DIR    = os.path.join(DATA_DIR, 'test_gnn')

# --- AWS ---
AWS_PROFILE   = 'Admin-Account-Access-541765610078'
AWS_REGION    = 'us-east-1'
ENDPOINT_NAME = 'fraud-detection-endpoint'

# --- Sampling ---
N_FRAUD_SAMPLES  = 15   # fraud transactions to visualize
N_LEGIT_SAMPLES  = 15   # legit transactions to visualize
MAX_1HOP         = 10   # max 1-hop merchants per anchor user
MAX_2HOP         = 5    # max 2-hop users per 1-hop merchant
FRAUD_THRESHOLD  = 0.5
RANDOM_SEED      = 42
np.random.seed(RANDOM_SEED)

print(f'Data dir:  {DATA_DIR}')
print(f'Test dir:  {TEST_DIR}')

Data dir:  /Users/atroyan/Projects/TW-sample-financial-fraud-detection-with-nvidia/data/TabFormer/gnn
Test dir:  /Users/atroyan/Projects/TW-sample-financial-fraud-detection-with-nvidia/data/TabFormer/gnn/test_gnn


## Load Data

In [16]:
print('Loading training graph...')
train_edges      = pd.read_csv(os.path.join(DATA_DIR, 'edges', 'user_to_merchant.csv'))
train_edge_attrs = pd.read_csv(os.path.join(DATA_DIR, 'edges', 'user_to_merchant_attr.csv'))
user_feats       = pd.read_csv(os.path.join(DATA_DIR, 'nodes', 'user.csv'))
merchant_feats   = pd.read_csv(os.path.join(DATA_DIR, 'nodes', 'merchant.csv'))

print('Loading test transactions...')
test_edges      = pd.read_csv(os.path.join(TEST_DIR, 'edges', 'user_to_merchant.csv'))
test_edge_attrs = pd.read_csv(os.path.join(TEST_DIR, 'edges', 'user_to_merchant_attr.csv'))
test_labels     = pd.read_csv(os.path.join(TEST_DIR, 'edges', 'user_to_merchant_label.csv'))

# Feature masks (group correlated features for Shapley coalitions)
user_mask     = pd.read_csv(os.path.join(TEST_DIR, 'nodes', 'user_feature_mask.csv'),     header=None).values.ravel().astype(np.int32)
merchant_mask = pd.read_csv(os.path.join(TEST_DIR, 'nodes', 'merchant_feature_mask.csv'), header=None).values.ravel().astype(np.int32)
edge_mask     = pd.read_csv(os.path.join(TEST_DIR, 'edges', 'user_to_merchant_feature_mask.csv'), header=None).values.ravel().astype(np.int32)

print()
print(f'Training: {len(train_edges):>7,} transactions | {len(user_feats):,} users | {len(merchant_feats):,} merchants')
print(f'Test:     {len(test_edges):>7,} transactions | Fraud rate: {test_labels.values.mean():.1%}')
print(f'Masks:    user={user_mask.shape} merchant={merchant_mask.shape} edge={edge_mask.shape}')

Loading training graph...
Loading test transactions...

Training: 301,524 transactions | 4,873 users | 42,942 merchants
Test:      25,803 transactions | Fraud rate: 8.1%
Masks:    user=(13,) merchant=(24,) edge=(38,)


In [17]:
print('Building neighbor index (user→merchants and merchant→users)...')

neighbors_user     = defaultdict(set)  # user_id -> set of merchant_ids
neighbors_merchant = defaultdict(set)  # merchant_id -> set of user_ids

for u, group in train_edges.groupby('src')['dst']:
    neighbors_user[u] = set(group.values)
for m, group in train_edges.groupby('dst')['src']:
    neighbors_merchant[m] = set(group.values)

user_degrees     = [len(v) for v in neighbors_user.values()]
merchant_degrees = [len(v) for v in neighbors_merchant.values()]

print(f'Unique users with history:     {len(neighbors_user):,}')
print(f'Unique merchants with history: {len(neighbors_merchant):,}')
print(f'Avg merchants per user:        {np.mean(user_degrees):.1f} (max {np.max(user_degrees)})')
print(f'Avg users per merchant:        {np.mean(merchant_degrees):.1f} (max {np.max(merchant_degrees)})')

Building neighbor index (user→merchants and merchant→users)...
Unique users with history:     4,873
Unique merchants with history: 42,942
Avg merchants per user:        49.5 (max 218)
Avg users per merchant:        5.6 (max 3337)


## 1. Graph Overview

A sample of the full training graph — bipartite layout with users on the left and merchants on the right.
Red edges are fraudulent transactions.

In [18]:
# Sample a balanced set of training edges for the overview
train_labels = pd.read_csv(os.path.join(DATA_DIR, 'edges', 'user_to_merchant_label.csv'))
fraud_train_idx = train_labels[train_labels['Fraud'] == 1].index
legit_train_idx = train_labels[train_labels['Fraud'] == 0].index

N_OVERVIEW = 150
sample_fraud = np.random.choice(fraud_train_idx, min(N_OVERVIEW // 2, len(fraud_train_idx)), replace=False)
sample_legit = np.random.choice(legit_train_idx, N_OVERVIEW // 2, replace=False)
overview_idx = np.sort(np.concatenate([sample_fraud, sample_legit]))

ov_edges  = train_edges.iloc[overview_idx]
ov_labels = train_labels.iloc[overview_idx]

# Collect unique nodes
ov_users     = sorted(ov_edges['src'].unique())
ov_merchants = sorted(ov_edges['dst'].unique())

user_pos_ov     = {u: (0.0, (i / max(len(ov_users) - 1, 1)) * 10) for i, u in enumerate(ov_users)}
merchant_pos_ov = {m: (2.0, (i / max(len(ov_merchants) - 1, 1)) * 10) for i, m in enumerate(ov_merchants)}

fig_overview = go.Figure()

# Draw edges (legit = light gray, fraud = red)
for color, label, idx_group in [('#E5E7EB', 'Legit Transaction', ov_labels[ov_labels['Fraud'] == 0].index),
                                  ('#EF4444', 'Fraud Transaction', ov_labels[ov_labels['Fraud'] == 1].index)]:
    x_lines, y_lines = [], []
    for idx in idx_group:
        row = ov_edges.loc[idx]
        u, m = int(row['src']), int(row['dst'])
        if u in user_pos_ov and m in merchant_pos_ov:
            ux, uy = user_pos_ov[u]
            mx, my = merchant_pos_ov[m]
            x_lines += [ux, mx, None]
            y_lines += [uy, my, None]
    fig_overview.add_trace(go.Scatter(
        x=x_lines, y=y_lines, mode='lines',
        line=dict(color=color, width=1.5),
        name=label, opacity=0.7,
    ))

# Draw user nodes
fig_overview.add_trace(go.Scatter(
    x=[user_pos_ov[u][0] for u in ov_users],
    y=[user_pos_ov[u][1] for u in ov_users],
    mode='markers',
    marker=dict(size=8, color='#1D4ED8', symbol='circle'),
    name='Users (card holders)',
    hovertext=[f'User {u}<br>Degree: {len(neighbors_user.get(u, set()))}' for u in ov_users],
    hoverinfo='text',
))

# Draw merchant nodes
fig_overview.add_trace(go.Scatter(
    x=[merchant_pos_ov[m][0] for m in ov_merchants],
    y=[merchant_pos_ov[m][1] for m in ov_merchants],
    mode='markers',
    marker=dict(size=8, color='#15803D', symbol='square'),
    name='Merchants',
    hovertext=[f'Merchant {m}<br>Degree: {len(neighbors_merchant.get(m, set()))}' for m in ov_merchants],
    hoverinfo='text',
))

fig_overview.add_annotation(text='Users', x=-0.1, y=10.5, xref='x', yref='y', showarrow=False,
                             font=dict(size=13, color='#1D4ED8'))
fig_overview.add_annotation(text='Merchants', x=2.1, y=10.5, xref='x', yref='y', showarrow=False,
                             font=dict(size=13, color='#15803D'))

fig_overview.update_layout(
    title=dict(text=f'Training Graph Sample ({N_OVERVIEW} transactions: {len(sample_fraud)} fraud, {N_OVERVIEW//2} legit)', x=0.5),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-0.3, 2.3]),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor='white',
    height=650,
    legend=dict(orientation='h', y=-0.05),
)
fig_overview.show()

## 2. Transaction Stream & Subgraph Extraction

We sample a stream of incoming test transactions (15 fraud + 15 legit), then for each transaction
extract its 2-hop ego subgraph from the training graph.

Use the **dropdown** to select a transaction and inspect its subgraph. The visualization shows:
- 🔵 **Anchor user** (large) — making the transaction
- 🟦 **2-hop users** (small) — others who share merchants with the anchor user
- 🟩 **Anchor merchant** (large) — receiving the transaction
- 🟢 **1-hop merchants** (small) — other merchants visited by the anchor user
- **Gray edges** — historical training transactions (context for the GNN)
- **Colored edge** — the new transaction being evaluated (red=fraud, green=legit)

In [19]:
# Sample a balanced set of test transactions
fraud_idx = test_labels[test_labels['Fraud'] == 1].index.tolist()
legit_idx  = test_labels[test_labels['Fraud'] == 0].index.tolist()

rng = np.random.default_rng(RANDOM_SEED)
chosen_fraud = rng.choice(fraud_idx, min(N_FRAUD_SAMPLES, len(fraud_idx)), replace=False)
chosen_legit = rng.choice(legit_idx, N_LEGIT_SAMPLES, replace=False)
sample_idx   = np.sort(np.concatenate([chosen_fraud, chosen_legit]))

sample_edges  = test_edges.iloc[sample_idx].reset_index(drop=True)
sample_attrs  = test_edge_attrs.iloc[sample_idx].reset_index(drop=True)
sample_labels = test_labels.iloc[sample_idx].reset_index(drop=True)

print(f'Sampled {len(sample_idx)} transactions:')
print(f'  Fraud: {int(sample_labels["Fraud"].sum())}')
print(f'  Legit: {int((1 - sample_labels["Fraud"]).sum())}')

Sampled 30 transactions:
  Fraud: 15
  Legit: 15


In [20]:
def extract_2hop_subgraph(anchor_user, anchor_merchant):
    """
    Extract the 2-hop ego subgraph around a transaction.

    Returns dict with:
      anchor_user, anchor_merchant
      all_users, all_merchants   — full node sets
      one_hop_merchants          — training history of anchor_user
      two_hop_users              — other users sharing 1-hop merchants
      context_edges              — training edges within the subgraph
    """
    # 1-hop: merchants anchor_user has visited in training
    visited = sorted(neighbors_user.get(anchor_user, set()))
    one_hop_merchants = [m for m in visited if m != anchor_merchant][:MAX_1HOP]
    all_merchants = sorted(set(one_hop_merchants) | {anchor_merchant})

    # 2-hop: other users who share those merchants
    two_hop_users = set()
    for m in one_hop_merchants:
        peers = sorted(neighbors_merchant.get(m, set()) - {anchor_user})
        two_hop_users.update(peers[:MAX_2HOP])
    two_hop_users = sorted(two_hop_users)
    all_users = sorted({anchor_user} | set(two_hop_users))

    # Context edges: training edges between subgraph nodes
    user_set     = set(all_users)
    merchant_set = set(all_merchants)
    mask = train_edges['src'].isin(user_set) & train_edges['dst'].isin(merchant_set)
    context_edges = train_edges[mask].copy()

    return {
        'anchor_user':      anchor_user,
        'anchor_merchant':  anchor_merchant,
        'all_users':        all_users,
        'all_merchants':    all_merchants,
        'one_hop_merchants': one_hop_merchants,
        'two_hop_users':    two_hop_users,
        'context_edges':    context_edges,
    }


print('Extracting subgraphs...')
subgraphs = []
for i, row in sample_edges.iterrows():
    sg = extract_2hop_subgraph(int(row['src']), int(row['dst']))
    subgraphs.append(sg)
    if (i + 1) % 5 == 0:
        print(f'  {i+1}/{len(sample_edges)} done')

print()
print('Subgraph stats (median):')
print(f'  Users in subgraph:    {int(np.median([len(s["all_users"]) for s in subgraphs]))}')
print(f'  Merchants in subgraph:{int(np.median([len(s["all_merchants"]) for s in subgraphs]))}')
print(f'  Context edges:        {int(np.median([len(s["context_edges"]) for s in subgraphs]))}')

Extracting subgraphs...
  5/30 done
  10/30 done
  15/30 done
  20/30 done
  25/30 done
  30/30 done

Subgraph stats (median):
  Users in subgraph:    22
  Merchants in subgraph:11
  Context edges:        214


In [21]:
def build_subgraph_traces(sg, gt_label, prediction=None, visible=False):
    """
    Build Plotly traces for a single subgraph visualization.
    Returns a list of traces and a layout annotation string.
    """
    anchor_user      = sg['anchor_user']
    anchor_merchant  = sg['anchor_merchant']
    all_users        = sg['all_users']
    all_merchants    = sg['all_merchants']
    one_hop_set      = set(sg['one_hop_merchants'])
    two_hop_set      = set(sg['two_hop_users'])
    context_edges    = sg['context_edges']

    # Node positions: bipartite layout
    n_u = len(all_users)
    n_m = len(all_merchants)
    user_pos     = {u: (0.0, (i / max(n_u - 1, 1)) * 10) for i, u in enumerate(all_users)}
    merchant_pos = {m: (2.0, (i / max(n_m - 1, 1)) * 10) for i, m in enumerate(all_merchants)}

    traces = []

    # -- Context edges (gray) --
    x_ctx, y_ctx = [], []
    for _, row in context_edges.iterrows():
        u, m = int(row['src']), int(row['dst'])
        if u in user_pos and m in merchant_pos:
            x_ctx += [user_pos[u][0], merchant_pos[m][0], None]
            y_ctx += [user_pos[u][1], merchant_pos[m][1], None]
    if x_ctx:
        traces.append(go.Scatter(
            x=x_ctx, y=y_ctx, mode='lines',
            line=dict(color='#D1D5DB', width=1),
            name='Historical transactions', visible=visible, showlegend=False,
            hoverinfo='none',
        ))

    # -- New transaction edge --
    if prediction is not None:
        edge_color = '#EF4444' if prediction > FRAUD_THRESHOLD else '#22C55E'
    else:
        edge_color = '#F59E0B'  # amber while waiting for prediction
    ux, uy = user_pos[anchor_user]
    mx, my = merchant_pos[anchor_merchant]
    traces.append(go.Scatter(
        x=[ux, mx], y=[uy, my], mode='lines',
        line=dict(color=edge_color, width=4),
        name='New transaction', visible=visible, showlegend=False,
        hoverinfo='none',
    ))

    # -- 2-hop users --
    hop2_us = [u for u in two_hop_set if u in user_pos]
    if hop2_us:
        traces.append(go.Scatter(
            x=[user_pos[u][0] for u in hop2_us],
            y=[user_pos[u][1] for u in hop2_us],
            mode='markers',
            marker=dict(size=10, color='#93C5FD', symbol='circle',
                        line=dict(color='#1D4ED8', width=1)),
            name='2-hop users', visible=visible, showlegend=False,
            hovertext=[f'User {u}<br>2-hop neighbor' for u in hop2_us],
            hoverinfo='text',
        ))

    # -- Anchor user --
    traces.append(go.Scatter(
        x=[user_pos[anchor_user][0]], y=[user_pos[anchor_user][1]],
        mode='markers+text',
        marker=dict(size=22, color='#1D4ED8', symbol='circle',
                    line=dict(color='white', width=2)),
        text=[f'U{anchor_user}'], textposition='middle left',
        name='Anchor user', visible=visible, showlegend=False,
        hovertext=[f'User {anchor_user}<br>Anchor — making this transaction<br>Training history: {len(neighbors_user.get(anchor_user, set()))} merchants'],
        hoverinfo='text',
    ))

    # -- 1-hop merchants --
    hop1_ms = [m for m in one_hop_set if m in merchant_pos]
    if hop1_ms:
        traces.append(go.Scatter(
            x=[merchant_pos[m][0] for m in hop1_ms],
            y=[merchant_pos[m][1] for m in hop1_ms],
            mode='markers',
            marker=dict(size=10, color='#86EFAC', symbol='square',
                        line=dict(color='#15803D', width=1)),
            name='1-hop merchants', visible=visible, showlegend=False,
            hovertext=[f'Merchant {m}<br>Previously visited by anchor user' for m in hop1_ms],
            hoverinfo='text',
        ))

    # -- Anchor merchant --
    traces.append(go.Scatter(
        x=[merchant_pos[anchor_merchant][0]], y=[merchant_pos[anchor_merchant][1]],
        mode='markers+text',
        marker=dict(size=22, color='#15803D', symbol='square',
                    line=dict(color='white', width=2)),
        text=[f'M{anchor_merchant}'], textposition='middle right',
        name='Anchor merchant', visible=visible, showlegend=False,
        hovertext=[f'Merchant {anchor_merchant}<br>Anchor — receiving this transaction<br>Total customers: {len(neighbors_merchant.get(anchor_merchant, set()))}'],
        hoverinfo='text',
    ))

    return traces


# Build all traces upfront (predictions not yet available — will be None)
all_traces = []
traces_per_tx = []

for i, sg in enumerate(subgraphs):
    gt = int(sample_labels.iloc[i]['Fraud'])
    t = build_subgraph_traces(sg, gt_label=gt, prediction=None, visible=(i == 0))
    traces_per_tx.append(len(t))
    all_traces.extend(t)

print(f'Total traces built: {len(all_traces)} across {len(subgraphs)} transactions')
print(f'(Avg {np.mean(traces_per_tx):.1f} traces per transaction)')

# Build dropdown buttons
total_traces = len(all_traces)
buttons = []
offset = 0
for i, sg in enumerate(subgraphs):
    gt    = int(sample_labels.iloc[i]['Fraud'])
    label = f"{'🚨' if gt else '✓'} Tx {i+1}  U{sg['anchor_user']}→M{sg['anchor_merchant']}"
    vis   = [False] * total_traces
    for j in range(traces_per_tx[i]):
        vis[offset + j] = True

    n_users     = len(sg['all_users'])
    n_merchants = len(sg['all_merchants'])
    n_ctx       = len(sg['context_edges'])
    title_text  = (f"Transaction {i+1}: User {sg['anchor_user']} → Merchant {sg['anchor_merchant']}  |  "
                   f"{'FRAUD' if gt else 'LEGIT'} (ground truth)  |  "
                   f"Subgraph: {n_users} users, {n_merchants} merchants, {n_ctx} context edges")

    buttons.append(dict(
        label=label,
        method='update',
        args=[{'visible': vis},
              {'title.text': title_text}],
    ))
    offset += traces_per_tx[i]

sg0 = subgraphs[0]
gt0 = int(sample_labels.iloc[0]['Fraud'])
initial_title = (f"Transaction 1: User {sg0['anchor_user']} → Merchant {sg0['anchor_merchant']}  |  "
                 f"{'FRAUD' if gt0 else 'LEGIT'} (ground truth)  |  "
                 f"Subgraph: {len(sg0['all_users'])} users, {len(sg0['all_merchants'])} merchants, "
                 f"{len(sg0['context_edges'])} context edges")

fig_subgraph = go.Figure(data=all_traces)
fig_subgraph.update_layout(
    title=dict(text=initial_title, x=0.02, font=dict(size=13)),
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        showactive=True,
        x=0.01, y=1.15, xanchor='left', yanchor='top',
        bgcolor='white', bordercolor='#D1D5DB',
    )],
    annotations=[
        dict(text='Select transaction:', x=0.01, y=1.2, xref='paper', yref='paper',
             showarrow=False, font=dict(size=12)),
        dict(text='<b>🔵 Anchor User</b>   🟦 2-hop Users   <b>🟩 Anchor Merchant</b>   🟢 1-hop Merchants   — History edges',
             x=0.5, y=-0.02, xref='paper', yref='paper',
             showarrow=False, font=dict(size=11, color='#4B5563')),
    ],
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-0.4, 2.4]),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor='#F9FAFB',
    height=680,
    margin=dict(t=120, b=60),
)
fig_subgraph.show()

Total traces built: 180 across 30 transactions
(Avg 6.0 traces per transaction)


## 3. Endpoint Inference

Send the sampled transactions to the live SageMaker endpoint as a single batch.
The payload includes node features for all referenced users and merchants,
the edge connectivity matrix, and transaction-level features.

In [22]:
def numpy_to_triton_inputs(data):
    dtype_map = {np.float32: 'FP32', np.int32: 'INT32', np.int64: 'INT64', np.bool_: 'BOOL'}
    return [
        {
            'name':     name,
            'shape':    list(arr.shape),
            'datatype': dtype_map.get(arr.dtype.type, 'FP32'),
            'data':     arr.flatten().tolist(),
        }
        for name, arr in data.items()
    ]


def build_batch_payload(edge_index_df, edge_attr_df, compute_shap=False):
    """
    Build an inference payload for a batch of transactions.
    Filters node features to only those referenced in the batch
    and remaps indices to consecutive local IDs.
    """
    srcs = edge_index_df['src'].values
    dsts = edge_index_df['dst'].values

    unique_users     = sorted(set(srcs))
    unique_merchants = sorted(set(dsts))

    user_to_local     = {u: i for i, u in enumerate(unique_users)}
    merchant_to_local = {m: i for i, m in enumerate(unique_merchants)}

    local_src = np.array([user_to_local[u] for u in srcs], dtype=np.int64)
    local_dst = np.array([merchant_to_local[m] for m in dsts], dtype=np.int64)

    return {
        'x_user':                          user_feats.iloc[unique_users].values.astype(np.float32),
        'x_merchant':                      merchant_feats.iloc[unique_merchants].values.astype(np.float32),
        'edge_index_user_to_merchant':     np.vstack([local_src, local_dst]),
        'edge_attr_user_to_merchant':      edge_attr_df.values.astype(np.float32),
        'COMPUTE_SHAP':                    np.array([compute_shap], dtype=np.bool_),
        'feature_mask_user':               user_mask,
        'feature_mask_merchant':           merchant_mask,
        'edge_feature_mask_user_to_merchant': edge_mask,
    }


print('Calling SageMaker endpoint...')
session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
runtime = session.client('sagemaker-runtime')

payload = build_batch_payload(sample_edges, sample_attrs)
body    = json.dumps({'inputs': numpy_to_triton_inputs(payload),
                      'outputs': [{'name': 'PREDICTION'}]})

import time
t0 = time.time()
response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Body=body,
)
elapsed = time.time() - t0

result      = json.loads(response['Body'].read())
predictions = np.array(result['outputs'][0]['data']).flatten()

print(f'Inference time: {elapsed:.2f}s  ({len(predictions) / elapsed:.0f} tx/sec)')
print(f'Predictions shape: {predictions.shape}')
print(f'Flagged as fraud: {(predictions > FRAUD_THRESHOLD).sum()} / {len(predictions)}')
print(f'Ground truth fraud: {int(sample_labels["Fraud"].sum())} / {len(sample_labels)}')

Calling SageMaker endpoint...
Inference time: 24.53s  (1 tx/sec)
Predictions shape: (30,)
Flagged as fraud: 2 / 30
Ground truth fraud: 15 / 30


## 4. Results Visualization

Per-transaction fraud scores compared against ground truth labels.

In [23]:
# Build results dataframe
results_df = pd.DataFrame({
    'tx':             range(1, len(predictions) + 1),
    'user':           sample_edges['src'].values,
    'merchant':       sample_edges['dst'].values,
    'fraud_score':    predictions,
    'predicted':      (predictions > FRAUD_THRESHOLD).astype(int),
    'ground_truth':   sample_labels['Fraud'].values,
    'n_users':        [len(sg['all_users']) for sg in subgraphs],
    'n_merchants':    [len(sg['all_merchants']) for sg in subgraphs],
    'n_context_edges':[len(sg['context_edges']) for sg in subgraphs],
})
results_df['correct'] = (results_df['predicted'] == results_df['ground_truth'])
results_df['outcome'] = results_df.apply(
    lambda r: ('TP' if r.predicted and r.ground_truth else
               'TN' if not r.predicted and not r.ground_truth else
               'FP' if r.predicted and not r.ground_truth else 'FN'), axis=1)

acc = results_df['correct'].mean()
print(f'Accuracy: {acc:.1%}')
print(results_df[['tx', 'user', 'merchant', 'fraud_score', 'predicted', 'ground_truth', 'outcome']].to_string(index=False))

Accuracy: 50.0%
 tx  user  merchant  fraud_score  predicted  ground_truth outcome
  1  1312       109     0.002306          0             0      TN
  2   746         7     0.172889          0             1      FN
  3  1616         9     0.005203          0             1      FN
  4   303        95     0.000148          0             1      FN
  5  1729       108     0.006269          0             0      TN
  6   308       161     0.007228          0             0      TN
  7  2793        23     0.000031          0             1      FN
  8  1347        64     0.107939          0             0      TN
  9   828        14     0.018581          0             0      TN
 10  2592       400     0.003428          0             0      TN
 11   146      1265     0.009501          0             1      FN
 12  1047         8     0.009262          0             1      FN
 13   769      3382     0.002043          0             0      TN
 14   128       170     0.942334          1             0   

In [24]:
COLOR_MAP = {'TP': '#EF4444', 'TN': '#22C55E', 'FP': '#F59E0B', 'FN': '#8B5CF6'}

fig_results = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Fraud Score Distribution',
        'Score vs Ground Truth (bubble = subgraph context edges)',
    ],
    column_widths=[0.4, 0.6],
)

# Left: histogram of fraud scores
for gt_val, gt_name, color in [(1, 'Fraud (ground truth)', '#EF4444'),
                                 (0, 'Legit (ground truth)', '#22C55E')]:
    scores = results_df[results_df['ground_truth'] == gt_val]['fraud_score']
    fig_results.add_trace(
        go.Histogram(x=scores, name=gt_name,
                     marker_color=color, opacity=0.7,
                     nbinsx=15, showlegend=True),
        row=1, col=1,
    )

# Threshold line
fig_results.add_vline(
    x=FRAUD_THRESHOLD, line_dash='dash', line_color='gray',
    annotation_text=f'threshold={FRAUD_THRESHOLD}',
    annotation_position='top right', row=1, col=1,
)

# Right: scatter (transaction index vs fraud score, colored by outcome)
for outcome, color in COLOR_MAP.items():
    sub = results_df[results_df['outcome'] == outcome]
    if len(sub) == 0:
        continue
    fig_results.add_trace(
        go.Scatter(
            x=sub['tx'],
            y=sub['fraud_score'],
            mode='markers',
            marker=dict(
                size=sub['n_context_edges'].clip(upper=200) / 10 + 8,
                color=color,
                opacity=0.85,
                line=dict(color='white', width=1),
            ),
            name=outcome,
            hovertext=[
                f"Tx {r.tx}: U{r.user}→M{r.merchant}<br>"
                f"Score: {r.fraud_score:.3f}<br>"
                f"Subgraph: {r.n_users} users, {r.n_merchants} merchants, {r.n_context_edges} edges"
                for _, r in sub.iterrows()
            ],
            hoverinfo='text',
        ),
        row=1, col=2,
    )

fig_results.add_hline(
    y=FRAUD_THRESHOLD, line_dash='dash', line_color='gray',
    annotation_text=f'threshold={FRAUD_THRESHOLD}',
    annotation_position='right', row=1, col=2,
)

fig_results.update_layout(
    title=dict(
        text=f'Inference Results — {len(predictions)} transactions  |  Accuracy: {acc:.1%}  |  '
             f'TP={int((results_df.outcome=="TP").sum())}  '
             f'TN={int((results_df.outcome=="TN").sum())}  '
             f'FP={int((results_df.outcome=="FP").sum())}  '
             f'FN={int((results_df.outcome=="FN").sum())}',
        x=0.5,
    ),
    barmode='overlay',
    height=500,
    legend=dict(orientation='h', y=-0.15),
    plot_bgcolor='white',
)
fig_results.update_xaxes(title_text='Fraud Score', row=1, col=1)
fig_results.update_yaxes(title_text='Count', row=1, col=1)
fig_results.update_xaxes(title_text='Transaction Index', row=1, col=2)
fig_results.update_yaxes(title_text='Fraud Score', range=[-0.05, 1.05], row=1, col=2)
fig_results.show()

In [25]:
# Re-draw the subgraph explorer now that we have predictions
# Each subgraph edge is colored by the model's prediction

all_traces_pred = []
traces_per_tx_pred = []

for i, (sg, pred) in enumerate(zip(subgraphs, predictions)):
    gt = int(sample_labels.iloc[i]['Fraud'])
    t = build_subgraph_traces(sg, gt_label=gt, prediction=float(pred), visible=(i == 0))
    traces_per_tx_pred.append(len(t))
    all_traces_pred.extend(t)

total_pred = len(all_traces_pred)
buttons_pred = []
offset = 0
for i, (sg, pred) in enumerate(zip(subgraphs, predictions)):
    gt     = int(sample_labels.iloc[i]['Fraud'])
    score  = float(pred)
    label  = f"{'🚨' if gt else '✓'} Tx {i+1}  U{sg['anchor_user']}→M{sg['anchor_merchant']}  [{score:.2f}]"
    vis    = [False] * total_pred
    for j in range(traces_per_tx_pred[i]):
        vis[offset + j] = True

    predicted = score > FRAUD_THRESHOLD
    correct   = predicted == bool(gt)
    status    = ('TP' if predicted and gt else 'TN' if not predicted and not gt
                 else 'FP' if predicted and not gt else 'FN')
    title_text = (
        f"Tx {i+1}: User {sg['anchor_user']} → Merchant {sg['anchor_merchant']}  |  "
        f"Score: {score:.3f}  |  Predicted: {'FRAUD' if predicted else 'LEGIT'}  |  "
        f"Ground truth: {'FRAUD' if gt else 'LEGIT'}  ({status})"
    )
    buttons_pred.append(dict(
        label=label, method='update',
        args=[{'visible': vis}, {'title.text': title_text}],
    ))
    offset += traces_per_tx_pred[i]

sg0   = subgraphs[0]
pred0 = float(predictions[0])
gt0   = int(sample_labels.iloc[0]['Fraud'])
init_title_pred = (
    f"Tx 1: User {sg0['anchor_user']} → Merchant {sg0['anchor_merchant']}  |  "
    f"Score: {pred0:.3f}  |  Predicted: {'FRAUD' if pred0 > FRAUD_THRESHOLD else 'LEGIT'}  |  "
    f"Ground truth: {'FRAUD' if gt0 else 'LEGIT'}"
)

fig_pred = go.Figure(data=all_traces_pred)
fig_pred.update_layout(
    title=dict(text=init_title_pred, x=0.02, font=dict(size=13)),
    updatemenus=[dict(
        buttons=buttons_pred,
        direction='down',
        showactive=True,
        x=0.01, y=1.18, xanchor='left', yanchor='top',
        bgcolor='white', bordercolor='#D1D5DB',
    )],
    annotations=[
        dict(text='Select transaction:', x=0.01, y=1.23, xref='paper', yref='paper',
             showarrow=False, font=dict(size=12)),
        dict(text='Edge color: 🔴 Fraud prediction   🟢 Legit prediction',
             x=0.5, y=-0.02, xref='paper', yref='paper',
             showarrow=False, font=dict(size=11, color='#4B5563')),
    ],
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-0.4, 2.4]),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor='#F9FAFB',
    height=680,
    margin=dict(t=130, b=60),
)
fig_pred.show()

## 5. Shapley Explanations

For the highest-scoring (most suspicious) transaction, request Shapley values.
Shapley values show which feature groups — user attributes, merchant attributes,
and transaction edge attributes — contributed most to the fraud prediction.

**Note:** Shapley computation is expensive (~10-30s). In production, only
request explanations for flagged transactions.

In [26]:
# Pick the top-scoring transaction (highest fraud probability)
top_idx = int(np.argmax(predictions))
top_score = float(predictions[top_idx])
top_gt = int(sample_labels.iloc[top_idx]['Fraud'])

print(f'Top fraud transaction: Tx {top_idx+1}')
print(f'  User {sample_edges.iloc[top_idx]["src"]} → Merchant {sample_edges.iloc[top_idx]["dst"]}')
print(f'  Fraud score: {top_score:.4f}')
print(f'  Ground truth: {"FRAUD" if top_gt else "LEGIT"}')
print()
print('Requesting Shapley values (this may take 15-30s)...')

# Build single-transaction payload with COMPUTE_SHAP=True
single_edge  = sample_edges.iloc[[top_idx]]
single_attrs = sample_attrs.iloc[[top_idx]]
payload_shap = build_batch_payload(single_edge, single_attrs, compute_shap=True)

body_shap = json.dumps({
    'inputs': numpy_to_triton_inputs(payload_shap),
    'outputs': [
        {'name': 'PREDICTION'},
        {'name': 'shap_values_user'},
        {'name': 'shap_values_merchant'},
        {'name': 'shap_values_user_to_merchant'},
    ],
})

t0 = time.time()
shap_response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Body=body_shap,
)
shap_elapsed = time.time() - t0

shap_result = json.loads(shap_response['Body'].read())
shap_outputs = {o['name']: np.array(o['data']).reshape(o['shape'])
                for o in shap_result['outputs']}

print(f'Shapley inference time: {shap_elapsed:.1f}s')
for name, arr in shap_outputs.items():
    print(f'  {name}: shape={arr.shape}  total_contribution={np.sum(arr):.4f}')

Top fraud transaction: Tx 14
  User 128 → Merchant 170
  Fraud score: 0.9423
  Ground truth: LEGIT

Requesting Shapley values (this may take 15-30s)...
Shapley inference time: 5.8s
  PREDICTION: shape=(1, 1)  total_contribution=0.9423
  shap_values_merchant: shape=(2,)  total_contribution=0.2962
  shap_values_user: shape=(1,)  total_contribution=0.4566
  shap_values_user_to_merchant: shape=(5,)  total_contribution=3.9017


In [27]:
# Shapley summary bar chart
# Each output is the total contribution of that feature group to the fraud score

feature_names = {
    'shap_values_user':              'User / Card features',
    'shap_values_merchant':          'Merchant features',
    'shap_values_user_to_merchant':  'Transaction edge features',
}

shap_totals = {}
for key, label in feature_names.items():
    if key in shap_outputs:
        shap_totals[label] = float(np.sum(shap_outputs[key]))

shap_df = pd.DataFrame(list(shap_totals.items()), columns=['Feature Group', 'Contribution'])
shap_df = shap_df.sort_values('Contribution')

colors = ['#EF4444' if v > 0 else '#3B82F6' for v in shap_df['Contribution']]

fig_shap = make_subplots(
    rows=1, cols=2,
    column_widths=[0.45, 0.55],
    subplot_titles=['Shapley Feature Group Contributions', 'Subgraph of Top Fraud Transaction'],
)

# Left: horizontal bar chart
fig_shap.add_trace(
    go.Bar(
        x=shap_df['Contribution'],
        y=shap_df['Feature Group'],
        orientation='h',
        marker_color=colors,
        text=[f"{v:+.4f}" for v in shap_df['Contribution']],
        textposition='outside',
        showlegend=False,
    ),
    row=1, col=1,
)
fig_shap.add_vline(x=0, line_color='gray', line_dash='solid', row=1, col=1)

# Right: subgraph of the top fraud transaction
top_sg = subgraphs[top_idx]
anchor_user     = top_sg['anchor_user']
anchor_merchant = top_sg['anchor_merchant']
all_users       = top_sg['all_users']
all_merchants   = top_sg['all_merchants']
one_hop_set     = set(top_sg['one_hop_merchants'])
two_hop_set     = set(top_sg['two_hop_users'])
context_edges   = top_sg['context_edges']

n_u = len(all_users)
n_m = len(all_merchants)
user_pos_top     = {u: (0.0, (i / max(n_u - 1, 1)) * 10) for i, u in enumerate(all_users)}
merchant_pos_top = {m: (1.5, (i / max(n_m - 1, 1)) * 10) for i, m in enumerate(all_merchants)}

# Context edges
x_c, y_c = [], []
for _, row in context_edges.iterrows():
    u, m = int(row['src']), int(row['dst'])
    if u in user_pos_top and m in merchant_pos_top:
        x_c += [user_pos_top[u][0], merchant_pos_top[m][0], None]
        y_c += [user_pos_top[u][1], merchant_pos_top[m][1], None]
if x_c:
    fig_shap.add_trace(go.Scatter(x=x_c, y=y_c, mode='lines',
                                   line=dict(color='#D1D5DB', width=1),
                                   showlegend=False, hoverinfo='none'), row=1, col=2)

# New transaction edge
ux, uy = user_pos_top[anchor_user]
mx, my = merchant_pos_top[anchor_merchant]
fig_shap.add_trace(go.Scatter(x=[ux, mx], y=[uy, my], mode='lines',
                               line=dict(color='#EF4444', width=4),
                               showlegend=False, hoverinfo='none'), row=1, col=2)

# 2-hop users
hop2 = [u for u in two_hop_set if u in user_pos_top]
if hop2:
    fig_shap.add_trace(go.Scatter(
        x=[user_pos_top[u][0] for u in hop2], y=[user_pos_top[u][1] for u in hop2],
        mode='markers', marker=dict(size=9, color='#93C5FD', symbol='circle',
                                     line=dict(color='#1D4ED8', width=1)),
        showlegend=False,
        hovertext=[f'User {u} (2-hop)' for u in hop2], hoverinfo='text',
    ), row=1, col=2)

# Anchor user
fig_shap.add_trace(go.Scatter(
    x=[user_pos_top[anchor_user][0]], y=[user_pos_top[anchor_user][1]],
    mode='markers+text',
    marker=dict(size=20, color='#1D4ED8', symbol='circle', line=dict(color='white', width=2)),
    text=[f'U{anchor_user}'], textposition='middle left',
    showlegend=False,
), row=1, col=2)

# 1-hop merchants
hop1 = [m for m in one_hop_set if m in merchant_pos_top]
if hop1:
    fig_shap.add_trace(go.Scatter(
        x=[merchant_pos_top[m][0] for m in hop1], y=[merchant_pos_top[m][1] for m in hop1],
        mode='markers', marker=dict(size=9, color='#86EFAC', symbol='square',
                                     line=dict(color='#15803D', width=1)),
        showlegend=False,
        hovertext=[f'Merchant {m} (1-hop)' for m in hop1], hoverinfo='text',
    ), row=1, col=2)

# Anchor merchant
fig_shap.add_trace(go.Scatter(
    x=[merchant_pos_top[anchor_merchant][0]], y=[merchant_pos_top[anchor_merchant][1]],
    mode='markers+text',
    marker=dict(size=20, color='#15803D', symbol='square', line=dict(color='white', width=2)),
    text=[f'M{anchor_merchant}'], textposition='middle right',
    showlegend=False,
), row=1, col=2)

fig_shap.update_layout(
    title=dict(
        text=(
            f"Shapley Explanation — Tx {top_idx+1}: "
            f"U{anchor_user}→M{anchor_merchant}  |  "
            f"Score: {top_score:.4f}  |  "
            f"{'FRAUD' if top_score > FRAUD_THRESHOLD else 'LEGIT'} predicted  |  "
            f"Ground truth: {'FRAUD' if top_gt else 'LEGIT'}"
        ),
        x=0.5, font=dict(size=13),
    ),
    height=520,
    plot_bgcolor='white',
)
fig_shap.update_xaxes(title_text='Shapley Contribution (→ fraud)', row=1, col=1)
fig_shap.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=2)
fig_shap.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=2)
fig_shap.add_annotation(
    text='Red = pushed toward fraud  |  Blue = pushed toward legit',
    x=0.22, y=-0.06, xref='paper', yref='paper',
    showarrow=False, font=dict(size=11, color='#4B5563'),
)
fig_shap.show()